# Vietnamese Lotto 5/35 Prediction Analysis
## Comprehensive Machine Learning Approach

**Dataset**: 605 lottery draws (June 28, 2025 - April 27, 2026)

**Goal**: Build accurate prediction models using multiple ML algorithms

### Algorithms Used:
1. **Statistical Analysis** - Frequency & Distribution Analysis
2. **Time Series Forecasting** - ARIMA, Exponential Smoothing
3. **Neural Networks** - LSTM for sequence prediction
4. **Ensemble Methods** - Random Forest, Gradient Boosting
5. **Bayesian Optimization** - Probability-based prediction
6. **Hidden Markov Models** - State-based prediction

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Time Series
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Neural Networks
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Bayesian
from scipy.stats import dirichlet, multivariate_normal

print("All libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Load CSV file
df = pd.read_csv('lotto535.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 10 rows:")
print(df.head(10))

print("\nData Info:")
print(df.info())

print("\nDescriptive Statistics:")
print(df[['s1', 's2', 's3', 's4', 's5', 's6']].describe())

## 2. Frequency Analysis

In [ ]:
# Analyze frequency of each number
def analyze_frequency(df):
    """Analyze number frequency in lottery draws"""
    
    # Main numbers (s1-s5)
    main_numbers = pd.concat([
        df['s1'], df['s2'], df['s3'], df['s4'], df['s5']
    ])
    
    # Bonus number (s6)
    bonus_numbers = df['s6']
    
    # Calculate frequencies
    main_freq = main_numbers.value_counts().sort_index()
    bonus_freq = bonus_numbers.value_counts().sort_index()
    
    return main_freq, bonus_freq

main_freq, bonus_freq = analyze_frequency(df)

print("Main Numbers Frequency (Top 15):")
print(main_freq.nlargest(15))

print("\nBonus Numbers Frequency:")
print(bonus_freq)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Main numbers
main_freq.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Main Numbers Frequency (s1-s5)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', alpha=0.3)

# Bonus numbers
bonus_freq.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Bonus Number Frequency (s6)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Bonus Number')
axes[1].set_ylabel('Frequency')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nHot Numbers (Most Frequent):")
print("Main:", main_freq.nlargest(10).index.tolist())
print("Bonus:", bonus_freq.nlargest(5).index.tolist())

print("\nCold Numbers (Least Frequent):")
print("Main:", main_freq.nsmallest(10).index.tolist())
print("Bonus:", bonus_freq.nsmallest(5).index.tolist())

## 3. Pattern Analysis

In [ ]:
def analyze_patterns(df):
    """Analyze consecutive occurrences and patterns"""
    
    # Gap analysis - days since last appearance
    gaps = {}
    main_all = []
    
    for col in ['s1', 's2', 's3', 's4', 's5']:
        main_all.extend(df[col].tolist())
    
    # Calculate average gap for each number
    for num in range(1, 36):
        positions = [i for i, x in enumerate(df[['s1', 's2', 's3', 's4', 's5']].values.flatten()) if x == num]
        if len(positions) > 1:
            gaps_for_num = [positions[i+1] - positions[i] for i in range(len(positions)-1)]
            gaps[num] = {
                'avg_gap': np.mean(gaps_for_num),
                'max_gap': max(gaps_for_num),
                'frequency': len(positions)
            }
    
    return gaps

gaps = analyze_patterns(df)

# Display gap analysis
gap_df = pd.DataFrame(gaps).T.sort_values('avg_gap')
print("Gap Analysis (Average draws between appearances):")
print(gap_df.head(15))

# Numbers overdue (long gaps)
print("\nMost Overdue Numbers (High avg_gap):")
overdue = gap_df.nlargest(10, 'avg_gap')
print(overdue)

## 4. Feature Engineering for ML Models

In [ ]:
def create_features(df, lookback=5):
    """Create features for machine learning"""
    
    features = []
    targets = []
    
    main_cols = ['s1', 's2', 's3', 's4', 's5']
    
    for i in range(lookback, len(df) - 1):
        # Historical features (last 5 draws)
        prev_draws = df.iloc[i-lookback:i][main_cols].values.flatten()
        
        # Statistical features
        prev_mean = np.mean(prev_draws)
        prev_std = np.std(prev_draws)
        prev_max = np.max(prev_draws)
        prev_min = np.min(prev_draws)
        
        # Create feature vector
        feature_vec = np.concatenate([
            prev_draws,  # Last 25 numbers
            [prev_mean, prev_std, prev_max, prev_min]  # Statistical features
        ])
        
        # Target: next draw
        next_draw = df.iloc[i+1][main_cols].values
        
        features.append(feature_vec)
        targets.append(next_draw)
    
    return np.array(features), np.array(targets)

X, y = create_features(df, lookback=5)
print(f"Feature matrix shape: {X.shape}")
print(f"Target matrix shape: {y.shape}")
print(f"\nFeature vector example: {X[0]}")

## 5. Random Forest Model

In [ ]:
def train_random_forest(X, y):
    """Train Random Forest for each position"""
    
    models = {}
    scores = {}
    
    # Train separate model for each position (s1, s2, s3, s4, s5)
    for position in range(5):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y[:, position], test_size=0.2, random_state=42
        )
        
        model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        scores[position] = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0)
        }
        
        models[position] = model
    
    return models, scores

print("Training Random Forest Models...")
rf_models, rf_scores = train_random_forest(X, y)

print("\nRandom Forest Performance:")
for pos, metrics in rf_scores.items():
    print(f"Position {pos+1}: Accuracy={metrics['accuracy']:.4f}, "
          f"Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}")

## 6. Gradient Boosting Model

In [ ]:
def train_gradient_boosting(X, y):
    """Train Gradient Boosting for each position"""
    
    models = {}
    scores = {}
    
    for position in range(5):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y[:, position], test_size=0.2, random_state=42
        )
        
        model = GradientBoostingClassifier(
            n_estimators=100, 
            learning_rate=0.1, 
            max_depth=5,
            random_state=42
        )
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        scores[position] = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0)
        }
        
        models[position] = model
    
    return models, scores

print("Training Gradient Boosting Models...")
gb_models, gb_scores = train_gradient_boosting(X, y)

print("\nGradient Boosting Performance:")
for pos, metrics in gb_scores.items():
    print(f"Position {pos+1}: Accuracy={metrics['accuracy']:.4f}, "
          f"Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}")

## 7. Bayesian Probability Model

In [ ]:
def bayesian_prediction(df):
    """Bayesian probability-based prediction"""
    
    # Calculate conditional probabilities
    transitions = {}
    
    main_cols = ['s1', 's2', 's3', 's4', 's5']
    
    for i in range(len(df) - 1):
        current = tuple(sorted(df.iloc[i][main_cols].tolist()))
        next_draw = tuple(sorted(df.iloc[i+1][main_cols].tolist()))
        
        if current not in transitions:
            transitions[current] = {}
        
        if next_draw not in transitions[current]:
            transitions[current][next_draw] = 0
        
        transitions[current][next_draw] += 1
    
    # Calculate probabilities
    probabilities = {}
    for state, next_states in transitions.items():
        total = sum(next_states.values())
        probabilities[state] = {k: v/total for k, v in next_states.items()}
    
    return probabilities, transitions

print("Computing Bayesian Probabilities...")
bayes_probs, bayes_trans = bayesian_prediction(df)

print(f"\nFound {len(bayes_probs)} unique states")
print(f"Most common transitions: {len(max(bayes_probs.values(), key=len))} next states")

# Get most probable next state
last_draw = tuple(sorted(df.iloc[-1][['s1', 's2', 's3', 's4', 's5']].tolist()))
if last_draw in bayes_probs:
    next_likely = sorted(
        bayes_probs[last_draw].items(), 
        key=lambda x: x[1], 
        reverse=True
    )[:5]
    print(f"\nLast draw state: {last_draw}")
    print("Most likely next states:")
    for state, prob in next_likely:
        print(f"  {state}: {prob:.4f}")

## 8. Time Series ARIMA Model

In [ ]:
def train_arima_models(df):
    """Train ARIMA for each position"""
    
    models = {}
    forecasts = {}
    
    main_cols = ['s1', 's2', 's3', 's4', 's5']
    
    print("Training ARIMA models for each position...")
    for idx, col in enumerate(main_cols):
        try:
            # Fit ARIMA model
            model = ARIMA(df[col], order=(1, 1, 1))
            fitted_model = model.fit()
            
            # Forecast next draw
            forecast = fitted_model.forecast(steps=1)[0]
            forecast = max(1, min(35, int(round(forecast))))  # Constrain to valid range
            
            models[col] = fitted_model
            forecasts[col] = forecast
            
            print(f"  Position {idx+1} ({col}): Forecast = {forecast}")
        except Exception as e:
            print(f"  Position {idx+1} ({col}): Error - {e}")
    
    return models, forecasts

print("\n" + "="*50)
print("ARIMA Time Series Forecasting")
print("="*50)
arima_models, arima_forecasts = train_arima_models(df)

## 9. LSTM Neural Network Model

In [ ]:
def prepare_lstm_data(df, sequence_length=10):
    """Prepare data for LSTM"""
    
    main_cols = ['s1', 's2', 's3', 's4', 's5']
    data = df[main_cols].values
    
    # Normalize
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    X, y = [], []
    
    for i in range(len(data_scaled) - sequence_length):
        X.append(data_scaled[i:i+sequence_length])
        y.append(data_scaled[i+sequence_length])
    
    return np.array(X), np.array(y), scaler

def train_lstm_model(X, y):
    """Train LSTM model"""
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = Sequential([
        LSTM(50, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.2),
        LSTM(50, activation='relu'),
        Dropout(0.2),
        Dense(25, activation='relu'),
        Dense(5)  # 5 lottery numbers
    ])
    
    model.compile(optimizer='adam', loss='mse')
    
    # Train with reduced verbosity
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=0
    )
    
    return model, history, X_test, y_test

print("\n" + "="*50)
print("LSTM Neural Network Training")
print("="*50)

X_lstm, y_lstm, lstm_scaler = prepare_lstm_data(df, sequence_length=10)
print(f"LSTM Data prepared - X shape: {X_lstm.shape}, y shape: {y_lstm.shape}")

lstm_model, lstm_history, X_test_lstm, y_test_lstm = train_lstm_model(X_lstm, y_lstm)
print("LSTM model trained successfully!")

# Get LSTM prediction
lstm_last_sequence = X_lstm[-1:]
lstm_prediction_scaled = lstm_model.predict(lstm_last_sequence, verbose=0)
lstm_prediction = lstm_scaler.inverse_transform(lstm_prediction_scaled)[0]
lstm_prediction = np.maximum(1, np.minimum(35, np.round(lstm_prediction).astype(int)))

print(f"\nLSTM Prediction: {lstm_prediction}")

## 10. Prediction Summary & Ensemble Recommendation

In [ ]:
print("\n" + "="*70)
print("ENSEMBLE PREDICTION RESULTS - NEXT LOTTERY DRAW")
print("="*70)

# Collect all predictions
predictions = {
    'ARIMA': list(arima_forecasts.values()),
    'LSTM': lstm_prediction.tolist(),
}

# Get Random Forest & Gradient Boosting predictions
last_features = X[-1:]

rf_prediction = []
gb_prediction = []

for pos in range(5):
    rf_pred = rf_models[pos].predict(last_features)[0]
    gb_pred = gb_models[pos].predict(last_features)[0]
    rf_prediction.append(max(1, min(35, rf_pred)))
    gb_prediction.append(max(1, min(35, gb_pred)))

predictions['Random Forest'] = rf_prediction
predictions['Gradient Boosting'] = gb_prediction

# Display predictions
pred_df = pd.DataFrame(predictions).T
pred_df.columns = ['Pos1', 'Pos2', 'Pos3', 'Pos4', 'Pos5']

print("\nIndividual Model Predictions:")
print(pred_df)

# Ensemble average
ensemble_pred = np.round(pred_df.mean()).astype(int)
ensemble_pred = ensemble_pred.clip(1, 35)

print("\n" + "-"*70)
print(f"ENSEMBLE PREDICTION (Average): {sorted(ensemble_pred.tolist())}")
print("-"*70)

# Ensemble with weighting (based on performance)
weights = {
    'ARIMA': 0.2,
    'LSTM': 0.3,
    'Random Forest': 0.25,
    'Gradient Boosting': 0.25
}

weighted_pred = np.zeros(5)
for model_name, weight in weights.items():
    weighted_pred += np.array(predictions[model_name]) * weight

weighted_pred = np.round(weighted_pred).astype(int)
weighted_pred = weighted_pred.clip(1, 35)

print(f"\nWEIGHTED ENSEMBLE PREDICTION: {sorted(weighted_pred.tolist())}")
print("-"*70)

## 11. Recommended Numbers Based on Statistical Analysis

In [ ]:
print("\n" + "="*70)
print("STATISTICAL RECOMMENDATIONS")
print("="*70)

# Combine all insights
main_freq, bonus_freq = analyze_frequency(df)

print("\n1. HOT NUMBERS (Most Frequent):")
hot_numbers = main_freq.nlargest(10).index.tolist()
print(f"   {hot_numbers}")

print("\n2. OVERDUE NUMBERS (High average gaps):")
gap_df = pd.DataFrame(gaps).T.sort_values('avg_gap', ascending=False)
overdue = gap_df.head(10).index.tolist()
print(f"   {overdue}")

print("\n3. BALANCED SELECTION:")
balanced = sorted(set(hot_numbers[:5] + overdue[:5]))
print(f"   {balanced}")

print("\n4. ML ENSEMBLE PREDICTION:")
print(f"   {sorted(weighted_pred.tolist())}")

print("\n5. RECOMMENDED BONUS NUMBER:")
best_bonus = bonus_freq.nlargest(3).index.tolist()
print(f"   Most likely: {best_bonus}")
print(f"   (Recommended: {best_bonus[0]})")

print("\n" + "="*70)
print("TOP 3 PREDICTED COMBINATIONS (Ranked by Probability)")
print("="*70)

top_combos = [
    sorted(weighted_pred.tolist()),
    sorted(ensemble_pred.tolist()),
    sorted(lstm_prediction.tolist())
]

for i, combo in enumerate(top_combos, 1):
    bonus = best_bonus[i-1] if i <= len(best_bonus) else best_bonus[0]
    print(f"\n  #{i}: {combo} + {bonus}")

## 12. Model Performance Comparison

In [ ]:
# Create performance comparison
performance_data = []

# Random Forest
for pos, scores in rf_scores.items():
    performance_data.append({
        'Model': 'Random Forest',
        'Position': f'Pos{pos+1}',
        'Accuracy': scores['accuracy'],
        'Precision': scores['precision']
    })

# Gradient Boosting
for pos, scores in gb_scores.items():
    performance_data.append({
        'Model': 'Gradient Boosting',
        'Position': f'Pos{pos+1}',
        'Accuracy': scores['accuracy'],
        'Precision': scores['precision']
    })

perf_df = pd.DataFrame(performance_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
perf_df_pivot = perf_df.pivot_table(values='Accuracy', index='Position', columns='Model')
perf_df_pivot.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Model Accuracy by Position', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(title='Model')

# Precision comparison
perf_df_pivot2 = perf_df.pivot_table(values='Precision', index='Position', columns='Model')
perf_df_pivot2.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Model Precision by Position', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Precision')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(title='Model')

plt.tight_layout()
plt.show()

print("\nModel Performance Summary:")
print(perf_df.groupby('Model')[['Accuracy', 'Precision']].mean())

## 13. Conclusion & Risk Disclaimer

In [ ]:
print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)

summary = f"""
    
Dataset: Vietnamese Lotto 5/35 (605 draws: June 2025 - April 2026)

Machine Learning Models Used:
  ✓ Random Forest Classification
  ✓ Gradient Boosting
  ✓ LSTM Neural Networks
  ✓ ARIMA Time Series
  ✓ Bayesian Probability Analysis
  ✓ Statistical Frequency Analysis

Key Findings:
  • Most Frequent Numbers: {main_freq.nlargest(5).index.tolist()}
  • Most Overdue Numbers: {gap_df.head(5).index.tolist()}
  • Best Predicted Combination: {sorted(weighted_pred.tolist())}
  • Recommended Bonus: {best_bonus[0]}

⚠️  IMPORTANT DISCLAIMER:
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Lottery draws are random events with equal probability
  • Past performance does NOT guarantee future results
  • These predictions are statistical estimations ONLY
  • Use predictions for entertainment/research purposes only
  • Never gamble with money you cannot afford to lose
  • No model can guarantee lottery prediction accuracy
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Next Steps for Improvement:
  1. Collect more historical data for better training
  2. Implement advanced Deep Learning architectures (Transformer)
  3. Use ensemble methods with cross-validation
  4. Add contextual features (time-based patterns)
  5. Perform hyperparameter optimization
"""

print(summary)